# Recipe 01 — Basic Pipeline
> Build a complete context window from scratch in under 20 lines.

| | |
|---|---|
| **Module** | `anchor.pipeline` |
| **Key classes** | `ContextPipeline`, `QueryBundle`, `SlidingWindowMemory`, `GenericTextFormatter` |
| **Difficulty** | Beginner |

In [ ]:
from anchor.pipeline import ContextPipeline
from anchor.models import QueryBundle, ContextItem, SourceType
from anchor.formatters import GenericTextFormatter
from anchor.memory import SlidingWindowMemory

## 1 — Create the pipeline
A `ContextPipeline` manages the full lifecycle of building a context window.
Set `max_tokens` to cap the total budget available for context.

In [ ]:
pipeline = ContextPipeline(max_tokens=4096)
print(f"Pipeline created with {pipeline.max_tokens} token budget")

## 2 — Add a system prompt
The system prompt is always placed first and counts against the token budget.

In [ ]:
pipeline.add_system_prompt("You are a helpful assistant.")
print("System prompt added")

## 3 — Attach a formatter
`GenericTextFormatter` converts context items into plain-text blocks
suitable for any LLM.

In [ ]:
pipeline.with_formatter(GenericTextFormatter())
print("Formatter attached")

## 4 — Wire up conversation memory
`SlidingWindowMemory` keeps the most recent turns that fit within its
own token budget.

In [ ]:
memory = SlidingWindowMemory(max_tokens=2048)
memory.add_turn("user", "What is context engineering?")
memory.add_turn("assistant", "Context engineering is the practice of "
                "curating the right information into an LLM's context window.")
pipeline.with_memory(memory)
print(f"Memory has {len(memory.get_turns())} turns")

## 5 — Build the context window
Call `pipeline.build()` with a `QueryBundle` to assemble the final window.

In [ ]:
query = QueryBundle(query_str="Tell me more about it")
result = pipeline.build(query)

print(f"Items in window : {len(result.window.items)}")
print(f"Tokens used     : {result.window.used_tokens}")
print(f"Utilization     : {result.window.utilization:.1%}")

## 6 — Inspect the formatted output
The `formatted_output` string is what you would pass to your LLM.

In [ ]:
output = result.formatted_output
preview = output[:300] if isinstance(output, str) else str(output)[:300]
print(preview)

## Key Takeaways
- `ContextPipeline(max_tokens=N)` is the entry point for every context build.
- Attach a **formatter**, **memory**, and optional **steps** before calling `build()`.
- `result.window` gives token-level visibility into what was included.

**Next:** [Built-in Steps](02_builtin_steps.ipynb)